In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("collisions.csv", dtype=str, low_memory=False)
raw_rows = len(df)
print(raw_rows)

2269187


In [2]:
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

# trim spaces in all text columns
for c in df.columns:
    df[c] = df[c].str.strip()

# drop duplicate crash IDs
print("Duplicate IDs:", df["collision_id"].duplicated().sum())
df = df.drop_duplicates(subset="collision_id")

# location column is only a copy of latitude + longitude
df = df.drop(columns=["location"])

Duplicate IDs: 0


In [3]:
df["crash_date"] = pd.to_datetime(df["crash_date"], format="%m/%d/%Y", errors="coerce")
t = pd.to_datetime(df["crash_time"], format="%H:%M", errors="coerce")
df["hour"] = t.dt.hour

df["year"] = df["crash_date"].dt.year
df["month"] = df["crash_date"].dt.month
df["month_name"] = df["crash_date"].dt.month_name()
df["day_of_week"] = df["crash_date"].dt.day_name()

print("Bad dates:", df["crash_date"].isna().sum(), " Bad times:", df["hour"].isna().sum())

Bad dates: 0  Bad times: 0


In [6]:
df[["crash_date", "hour", "year", "day_of_week"]].head(15)
print("Shape:", df.shape)

Shape: (2269187, 33)


In [ ]:
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

# invalid or outside NYC -> NULL (keep the crash row)
bad = ~(df["latitude"].between(40.4, 41.0) & df["longitude"].between(-74.3, -73.6))
df.loc[bad, ["latitude", "longitude"]] = np.nan

df["borough"] = df["borough"].str.upper().fillna("UNKNOWN")
df["zip_code"] = df["zip_code"].fillna("UNKNOWN")

In [8]:
print(df["borough"].value_counts())
print("Missing lat/long:", df["latitude"].isna().sum())

borough
UNKNOWN          691375
BROOKLYN         506806
QUEENS           422409
MANHATTAN        348485
BRONX            234088
STATEN ISLAND     66024
Name: count, dtype: int64
Missing lat/long: 248546


In [9]:
num_cols = [c for c in df.columns if c.startswith("number_of_")]
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce").fillna(0).astype(int)
df["total_casualties"] = df["number_of_persons_injured"] + df["number_of_persons_killed"]

In [10]:
print(df[num_cols].dtypes)
print(df[num_cols].sum())
print(df["total_casualties"].describe())

number_of_persons_injured        int64
number_of_persons_killed         int64
number_of_pedestrians_injured    int64
number_of_pedestrians_killed     int64
number_of_cyclist_injured        int64
number_of_cyclist_killed         int64
number_of_motorist_injured       int64
number_of_motorist_killed        int64
dtype: object
number_of_persons_injured        756353
number_of_persons_killed           3617
number_of_pedestrians_injured    137819
number_of_pedestrians_killed       1805
number_of_cyclist_injured         67336
number_of_cyclist_killed            294
number_of_motorist_injured       540227
number_of_motorist_killed          1466
dtype: int64
count    2.269187e+06
mean     3.349085e-01
std      7.204998e-01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      4.300000e+01
Name: total_casualties, dtype: float64


In [11]:
df["severity"] = np.select(
    [df["number_of_persons_killed"] > 0, df["number_of_persons_injured"] > 0],
    ["Fatal", "Injury"],
    default="Property damage only",
)

In [12]:
print(df["severity"].value_counts())
print("Fatal crashes check:", (df["number_of_persons_killed"] > 0).sum())

severity
Property damage only    1707973
Injury                   557745
Fatal                      3469
Name: count, dtype: int64
Fatal crashes check: 3469


In [13]:
factor_cols = [f"contributing_factor_vehicle_{i}" for i in range(1, 6)]
for c in factor_cols:
    df[c] = df[c].fillna("Unspecified").str.title().replace({"": "Unspecified"})

veh_cols = [f"vehicle_type_code_{i}" for i in range(1, 6)]
for c in veh_cols:
    df[c] = df[c].fillna("UNKNOWN").str.upper()

In [14]:
print(df["contributing_factor_vehicle_1"].nunique(), "unique factors")
print(df["contributing_factor_vehicle_1"].value_counts().head(15))

print(df["vehicle_type_code_1"].nunique(), "unique vehicle types")
print(df["vehicle_type_code_1"].value_counts().head(30))

64 unique factors
contributing_factor_vehicle_1
Unspecified                       764845
Driver Inattention/Distraction    462105
Failure To Yield Right-Of-Way     136046
Following Too Closely             121711
Backing Unsafely                   82577
Other Vehicular                    71429
Passing Or Lane Usage Improper     65544
Passing Too Closely                58028
Turning Improperly                 55751
Fatigued/Drowsy                    47599
Unsafe Lane Changing               44743
Traffic Control Disregarded        41898
Driver Inexperience                36928
Unsafe Speed                       35562
Alcohol Involvement                26503
Name: count, dtype: int64
1328 unique vehicle types
vehicle_type_code_1
SEDAN                                  660186
STATION WAGON/SPORT UTILITY VEHICLE    518578
PASSENGER VEHICLE                      416206
SPORT UTILITY / STATION WAGON          180291
TAXI                                    89190
PICK-UP TRUCK                      

In [17]:
def veh_group(v):
    if v in ("UNKNOWN", "UNK", ""):
        return "Unknown"
    if any(k in v for k in ["MOTORCYCLE", "MOTORBIKE", "MOPED", "MOTORSCOOTER"]):
        return "Motorcycle / Moped"
    if any(k in v for k in ["BIKE", "BICYCLE", "BICY", "E-BIK", "SCOOT", "E-SCO",
                            "STANDING", "PEDICAB"]):
        return "Bike / E-bike / Scooter"
    if any(k in v for k in ["AMBULANCE", "AMBU", "FIRE", "FDNY", "POLICE", "POLI"]):
        return "Emergency vehicle"
    if any(k in v for k in ["TAXI", "LIVERY", "UBER", "LYFT", "FOR HIRE", "LIMO"]):
        return "Taxi / For-hire"
    if "BUS" in v:
        return "Bus"
    if v == "PK" or any(k in v for k in ["TRUCK", "TRACT", "VAN", "DUMP", "FLAT", "PICK",
                                         "COM VEH", "BOX", "GARBAGE", "REFUSE", "CARRY",
                                         "TANK", "LIFT", "CEMENT", "CONCRETE", "TRAIL",
                                         "STAKE", "CHASSIS", "SNOW", "DELV"]):
        return "Truck / Van / Commercial"
    if any(k in v for k in ["SPORT UTILITY", "STATION WAGON", "SUV", "SUBN"]):
        return "SUV / Station wagon"
    if any(k in v for k in ["SEDAN", "PASSENGER", "CONVERTIBLE", "2 DR", "4 DR", "3-DOOR"]):
        return "Passenger car / Sedan"
    return "Other"

mapping = {v: veh_group(v) for v in df["vehicle_type_code_1"].unique()}
df["vehicle_group_1"] = df["vehicle_type_code_1"].map(mapping)

In [18]:
print(df["vehicle_group_1"].value_counts())

print(df.loc[df["vehicle_group_1"] == "Other", "vehicle_type_code_1"].value_counts().head(20))

vehicle_group_1
Passenger car / Sedan       1127810
SUV / Station wagon          699127
Truck / Van / Commercial     178916
Taxi / For-hire              100111
Bus                           39773
Unknown                       37142
Bike / E-bike / Scooter       27995
Other                         27752
Motorcycle / Moped            19567
Emergency vehicle             10994
Name: count, dtype: int64
vehicle_type_code_1
OTHER                    22973
UTIL                       172
MULTI-WHEELED VEHICLE      121
TRAC                        93
MOPD                        90
USPS                        89
COMMERCIAL                  88
COM                         86
UNKNO                       86
OPEN BODY                   85
SCHOO                       84
BULK AGRICULTURE            81
EBIK                        79
DELIV                       77
UN/C                        75
COMME                       74
TOW T                       74
ESST                        72
UTILI               

In [19]:
extra = {
    "TRAC": "Truck / Van / Commercial", "COMMERCIAL": "Truck / Van / Commercial",
    "COM": "Truck / Van / Commercial", "COMME": "Truck / Van / Commercial",
    "TOW T": "Truck / Van / Commercial", "OPEN BODY": "Truck / Van / Commercial",
    "PALLET": "Truck / Van / Commercial", "DELIV": "Truck / Van / Commercial",
    "USPS": "Truck / Van / Commercial",
    "MOPD": "Motorcycle / Moped", "EBIK": "Bike / E-bike / Scooter",
    "SCHOO": "Bus",
    "UNKNO": "Unknown", "UN/C": "Unknown",
}
mapping.update(extra)
df["vehicle_group_1"] = df["vehicle_type_code_1"].map(mapping)
print(df["vehicle_group_1"].value_counts())

vehicle_group_1
Passenger car / Sedan       1127810
SUV / Station wagon          699127
Truck / Van / Commercial     179640
Taxi / For-hire              100111
Bus                           39857
Unknown                       37303
Bike / E-bike / Scooter       28074
Other                         26614
Motorcycle / Moped            19657
Emergency vehicle             10994
Name: count, dtype: int64


In [20]:
# ---------- 1. Row count check ----------
print("Raw rows  :", raw_rows)
print("Clean rows:", len(df))
print("Rows removed:", raw_rows - len(df))

# ---------- 2. Totals check (must match Cell 5) ----------
print("\nTotal injured:", df["number_of_persons_injured"].sum())
print("Total killed :", df["number_of_persons_killed"].sum())

# ---------- 3. Key rules that must be true ----------
assert df["collision_id"].is_unique, "Duplicate collision IDs found"
assert df["crash_date"].notna().all(), "Some crash dates are missing"
assert df["hour"].between(0, 23).all(), "Some hours are outside 0-23"
assert df["severity"].isin(["Fatal", "Injury", "Property damage only"]).all()
print("\nAll checks passed")

# ---------- 4. Summary of the final table ----------
print("\nShape:", df.shape)
print("Date range:", df["crash_date"].min().date(), "to", df["crash_date"].max().date())
print("\nCrashes per year:")
print(df["year"].value_counts().sort_index())
print("\nMissing values (columns that still have blanks):")
print(df.isna().sum()[lambda s: s > 0])

# ---------- 5. Save ----------
df.to_csv("collisions_clean.csv", index=False)
print("\nSaved collisions_clean.csv")

Raw rows  : 2269187
Clean rows: 2269187
Rows removed: 0

Total injured: 756353
Total killed : 3617

All checks passed

Shape: (2269187, 36)
Date range: 2012-07-01 to 2026-06-11

Crashes per year:
year
2012    100545
2013    203742
2014    206046
2015    217708
2016    229833
2017    231007
2018    231564
2019    211486
2020    112918
2021    110558
2022    103887
2023     96607
2024     91316
2025     85546
2026     36424
Name: count, dtype: int64

Missing values (columns that still have blanks):
latitude              248546
longitude             248546
on_street_name        499179
cross_street_name     870235
off_street_name      1862382
dtype: int64

Saved collisions_clean.csv


In [21]:
df.to_csv("collisions_clean.csv", index=False)
